# Fine-tuning local con Unsloth: Qwen3.5-4B y Gemma 4 E2B

**Kai — La comunidad AI de LATAM** · [kaizen.la](https://kaizen.la) · [GitHub repo](https://github.com/MartinCalderon-x/kai-ml-notebooks)

Este notebook implementa fine-tuning LoRA de dos modelos sobre un **T4 de Google Colab gratuito** (15GB VRAM). El resultado es un archivo GGUF que podés correr offline en tu Mac M1, Ollama o Unsloth Studio.

**Antes de empezar:** `Entorno de ejecución → Cambiar tipo de entorno → T4 GPU`

| Modelo | VRAM necesaria | Caso de uso |
|--------|---------------|-------------|
| **Qwen3.5-4B** | ~10GB (bf16 LoRA) | Multilingüe, excelente en español |
| **Gemma 4 E2B** | ~8GB (4-bit LoRA) | Compacto, eficiente, visión opcional |

## Paso 0 — Verificar GPU

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No hay GPU. Andá a: Entorno de ejecución → Cambiar tipo de entorno → T4 GPU")

print(f"GPU   : {torch.cuda.get_device_name(0)}")
print(f"VRAM  : {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
print(f"CUDA  : {torch.version.cuda}")
print(f"Python: {__import__('sys').version.split()[0]}")

## Paso 1 — Instalar Unsloth

> **Importante**: instalamos `unsloth` y `unsloth_zoo` desde PyPI para garantizar versiones compatibles entre sí. Mezclar una versión de git con una de PyPI causa `ImportError`.

In [ ]:
%%capture
import sys

# Instalar unsloth + unsloth_zoo desde PyPI (versiones sincronizadas)
!{sys.executable} -m pip install unsloth unsloth_zoo --quiet

# transformers v5 es obligatorio para Qwen3.5
!{sys.executable} -m pip install "transformers>=5.0" datasets --quiet

## Paso 2 — Importar Unsloth PRIMERO

> Unsloth parchea PyTorch internamente. Debe importarse **antes** que `trl`, `transformers` o `peft`, de lo contrario el training es más lento y puede dar warnings.

In [ ]:
# ← Siempre el primer import
import unsloth
print(f"Unsloth version: {unsloth.__version__}")

## Paso 3 — Elegir modelo y cargar

In [ ]:
# ─── CONFIGURACIÓN ────────────────────────────────────────────────────────
MODELO = "qwen35"   # Opciones: "qwen35" | "gemma4"
MAX_SEQ_LENGTH = 2048
# ──────────────────────────────────────────────────────────────────────────

if MODELO == "qwen35":
    from unsloth import FastLanguageModel

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = "unsloth/Qwen3.5-4B-Instruct",
        max_seq_length = MAX_SEQ_LENGTH,
        load_in_4bit = False,
        load_in_16bit = True,   # QLoRA (4-bit) NO recomendado para Qwen3.5
        full_finetuning = False,
    )
    model = FastLanguageModel.get_peft_model(
        model,
        r = 16,
        target_modules = ["q_proj","k_proj","v_proj","o_proj",
                          "gate_proj","up_proj","down_proj"],
        lora_alpha = 16,
        lora_dropout = 0,
        bias = "none",
        use_gradient_checkpointing = "unsloth",
        random_state = 3407,
        max_seq_length = MAX_SEQ_LENGTH,
    )
    print("✅ Qwen3.5-4B-Instruct cargado con LoRA r=16")

elif MODELO == "gemma4":
    from unsloth import FastModel

    model, tokenizer = FastModel.from_pretrained(
        model_name = "unsloth/gemma-4-E2B-it",
        max_seq_length = MAX_SEQ_LENGTH,
        load_in_4bit = True,
        full_finetuning = False,
    )
    model = FastModel.get_peft_model(
        model,
        finetune_vision_layers = False,
        finetune_language_layers = True,
        finetune_attention_modules = True,
        finetune_mlp_modules = True,
        r = 8,
        lora_alpha = 8,
        lora_dropout = 0,
        bias = "none",
        random_state = 3407,
    )
    print("✅ Gemma 4 E2B cargado con LoRA r=8")

else:
    raise ValueError(f"MODELO='{MODELO}' no reconocido. Usá 'qwen35' o 'gemma4'")

## Paso 4 — Dataset

Elegí una opción:

| Opción | Descripción | Idioma |
|--------|-------------|--------|
| **A** | Demo (chip2) — carga inmediata, verifica el pipeline | Inglés |
| **B** | Tu propio JSONL desde Google Drive | El que quieras |
| **C** | Alpaca en español (HuggingFace) | Español |

In [ ]:
from datasets import load_dataset

# ─── ELEGÍ UNA OPCIÓN ─────────────────────────────────────────────────────
OPCION_DATASET = "A"   # "A" | "B" | "C"
# ──────────────────────────────────────────────────────────────────────────

if OPCION_DATASET == "A":
    url = "https://huggingface.co/datasets/laion/OIG/resolve/main/unified_chip2.jsonl"
    dataset = load_dataset("json", data_files={"train": url}, split="train[:500]")
    dataset_text_field = "text"
    print(f"✅ Dataset demo: {len(dataset)} ejemplos")
    print(f"Ejemplo: {dataset[0]['text'][:200]}...")

elif OPCION_DATASET == "B":
    from google.colab import drive
    drive.mount('/content/drive')
    RUTA = "/content/drive/MyDrive/mi-dataset.jsonl"   # ← cambiar
    dataset = load_dataset("json", data_files={"train": RUTA}, split="train")
    dataset_text_field = "text"                         # ← cambiar si es necesario
    print(f"✅ Dataset propio: {len(dataset)} ejemplos")

elif OPCION_DATASET == "C":
    dataset = load_dataset("bertin-project/alpaca-spanish", split="train[:1000]")
    def fmt(ex):
        return {"text": f"### Instrucción:\n{ex['instruction']}\n\n### Respuesta:\n{ex['output']}"}
    dataset = dataset.map(fmt)
    dataset_text_field = "text"
    print(f"✅ Alpaca español: {len(dataset)} ejemplos")

## Paso 5 — Entrenamiento

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    args = SFTConfig(
        dataset_text_field = dataset_text_field,
        max_seq_length = MAX_SEQ_LENGTH,
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 4,
        warmup_steps = 10,
        max_steps = 100,           # ~5 min en T4; reemplazá por num_train_epochs=1 para entrenamiento completo
        learning_rate = 2e-4,
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = f"outputs_{MODELO}",
        dataset_num_proc = 2,
        report_to = "none",
    ),
)

stats = trainer.train()
print(f"\n✅ Listo — Loss: {stats.metrics['train_loss']:.4f} — Tiempo: {stats.metrics['train_runtime']:.0f}s")

## Paso 6 — Test de inferencia

In [ ]:
from unsloth.chat_templates import get_chat_template

if MODELO == "qwen35":
    FastLanguageModel.for_inference(model)
    tokenizer = get_chat_template(tokenizer, chat_template="qwen-2.5")
else:
    FastModel.for_inference(model)
    tokenizer = get_chat_template(tokenizer, chat_template="gemma-4")

messages = [{"role": "user", "content": "Explicá en 3 pasos cómo funciona LoRA para fine-tuning de LLMs."}]

inputs = tokenizer.apply_chat_template(
    messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
).to("cuda")

with torch.no_grad():
    out = model.generate(inputs, max_new_tokens=512, temperature=0.7, do_sample=True)

print(tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True))

## Paso 7 — Exportar a GGUF y descargar

In [ ]:
import os, glob
from google.colab import files

OUTPUT_DIR = f"kai-{MODELO}-finetuned"
QUANT = "q4_k_m"   # q4_k_m (recomendado) | q8_0 (más preciso) | f16 (sin cuantización)

print(f"Exportando a GGUF ({QUANT})...")
model.save_pretrained_gguf(OUTPUT_DIR, tokenizer, quantization_method=QUANT)

for f in os.listdir(OUTPUT_DIR):
    sz = os.path.getsize(f"{OUTPUT_DIR}/{f}") / 1024**2
    print(f"  {f}  ({sz:.0f} MB)")

# Descargar a tu Mac
for gguf in glob.glob(f"{OUTPUT_DIR}/*.gguf"):
    print(f"Descargando {gguf}...")
    files.download(gguf)

print(f"""
✅ Descargado. En tu Mac:
  echo 'FROM ./{OUTPUT_DIR}-{QUANT}.gguf' > Modelfile
  ollama create kai-{MODELO} -f Modelfile
  ollama run kai-{MODELO}
""")

## Paso 8 — (Opcional) Subir a HuggingFace Hub

In [ ]:
# Descomentar y completar para publicar en HuggingFace

# HF_TOKEN    = "hf_..."       # ← tu token en hf.co/settings/tokens
# HF_USERNAME = "tu-usuario"

# model.push_to_hub_gguf(
#     f"{HF_USERNAME}/kai-{MODELO}-finetuned",
#     tokenizer,
#     quantization_method = QUANT,
#     token = HF_TOKEN,
# )

print("(celda comentada — descomentar para publicar en HuggingFace)")

---

## Recursos

- [Unsloth Studio](https://unsloth.ai/docs/new/studio) — correr el GGUF en tu Mac sin código
- [Unsloth Qwen3.5](https://unsloth.ai/docs/models/qwen3.5/fine-tune) — docs oficiales
- [Unsloth Gemma 4](https://unsloth.ai/docs/models/gemma-4/train) — docs oficiales
- [Kai LATAM](https://kaizen.la) — comunidad AI en español

---

*Notebook por [Kai](https://kaizen.la). Basado en documentación oficial de [Unsloth](https://github.com/unslothai/unsloth) (MIT License).*